[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C64_ML_Knowledge_QA_Course/03_architectures/03_architectures_qa.ipynb)

# 03 · 深度学习架构问答（卷积口算 / 1×1与深度可分离 / gridding / BN小batch / attention √d_k / 位置编码外推）

目标：把「说说卷积和 Transformer 的区别」这类问题，从背概念变成**能亲手算出数字来支撑答案**。

本 notebook 你会亲手实现并验证：
1. **参数量/FLOPs 计算器 + 感受野递推**，对照真实 backbone 的层配置校验
2. **1×1 卷积三作用**的数值演示（bottleneck 参数节省的具体倍数）与**深度可分离卷积**的收益公式
3. **空洞卷积的 gridding artifact**：用集合论证明"某些像素永远不会被采样到"
4. **BN vs GroupNorm 在小 batch 下的统计稳定性**对比（含 GN=LN/IN 特例的验证）
5. **从零实现 scaled dot-product attention**，用数值实验验证 $\sqrt{d_k}$ 缩放的必要性
6. **三类位置编码的外推行为**对比（固定正弦 / 可学习绝对 / 类 RoPE 相对）

> 心智模型：**架构问答的"追问陷阱"几乎都藏在一个可以用 20 行 numpy 复现的具体数字里——本 notebook 就是把这些数字都跑出来。**

## 0 · 环境自检

In [ ]:
import sys
import numpy as np

print('Python:', sys.version.split()[0])
print('numpy :', np.__version__)
assert sys.version_info >= (3, 8)
rng = np.random.default_rng(0)
print('\n✅ 环境自检通过：本课全程 numpy + 标准库，无需 GPU，不联网。')

## 1 · 参数量、FLOPs 与感受野递推：对照真实 backbone 校验

In [ ]:
def conv_params(cin, cout, k, bias=False):
    p = k * k * cin * cout
    return p + cout if bias else p

def conv_macs(cin, cout, k, hout, wout):
    return k * k * cin * cout * hout * wout

def out_size(hin, k, s, p):
    return (hin + 2 * p - k) // s + 1

# ResNet 系列的 stem 层：7x7, Cin=3, Cout=64, stride=2, pad=3，输入 224x224（无 bias，后接 BN）
p_stem = conv_params(3, 64, 7, bias=False)
h_out = out_size(224, 7, 2, 3)
macs_stem = conv_macs(3, 64, 7, h_out, h_out)

print(f'ResNet stem conv1: 参数量={p_stem:,}  输出分辨率={h_out}x{h_out}  MACs={macs_stem:,}  FLOPs(=2xMACs)={2*macs_stem:,}')
assert p_stem == 9408, 'ResNet conv1 是一个广为人知的校验数字：7*7*3*64=9408'
assert h_out == 112
assert macs_stem == 118_013_952
print('\n✅ 与公开资料里 ResNet conv1 的参数量/MACs 数字一致，说明公式没写错。')

In [ ]:
def receptive_field(layers):
    """layers: [(k, s), ...] -> [(RF, jump), ...]，RF_0=1, jump_0=1。"""
    rf, jump = 1, 1
    out = [(rf, jump)]
    for k, s in layers:
        rf = rf + (k - 1) * jump
        jump = jump * s
        out.append((rf, jump))
    return out

# 三层 3x3 stride1 堆叠 == 一个 7x7 卷积的感受野（VGG 论文的经典论证）
seq = receptive_field([(3, 1), (3, 1), (3, 1)])
print('三层 3x3 s1 堆叠的 RF 序列:', seq)
assert seq[-1][0] == 7, '三层 3x3 堆叠应等价一个 7x7 卷积的感受野'

# 参数量对比：三层 3x3 堆叠 vs 一个 7x7，同样感受野
p_stack = 3 * conv_params(64, 64, 3)
p_single = conv_params(64, 64, 7)
print(f'三层 3x3 堆叠参数量: {p_stack:,}   单层 7x7 参数量: {p_single:,}   节省比例: {1 - p_stack/p_single:.2%}')
assert p_stack < p_single

# ResNet stem + maxpool + 两层 3x3：验证感受野随层数增长
seq2 = receptive_field([(7, 2), (3, 2), (3, 1), (3, 1)])
print('ResNet stem(7x7,s2) + pool(3x3,s2) + 两层3x3(s1) 的 RF 序列:', seq2)
assert seq2[-1][0] == 27
print('\n✅ 验证：感受野递推公式在真实层配置下算得出确定的数字，可以在白板上现推。')

## 2 · 1×1 卷积三作用 + 深度可分离卷积的收益公式

In [ ]:
# ---- 1x1 卷积不贡献感受野（作用①：只做通道变换，不做空间混合）----
rf_before = receptive_field([(3, 1)])[-1]
rf_after_1x1 = receptive_field([(3, 1), (1, 1)])[-1]
print('叠加 3x3 后的 RF:', rf_before, '  再叠加 1x1 后的 RF:', rf_after_1x1)
assert rf_before[0] == rf_after_1x1[0], '1x1 卷积(k=1)不应改变感受野'

# ---- bottleneck (256->64->64->256) vs 两层 3x3 堆叠 (256->256->256) ----
bott = conv_params(256, 64, 1) + conv_params(64, 64, 3) + conv_params(64, 256, 1)
plain = conv_params(256, 256, 3) + conv_params(256, 256, 3)
print(f'\nBottleneck(1x1+3x3+1x1) 参数量: {bott:,}')
print(f'两层 3x3 堆叠(256->256->256) 参数量: {plain:,}')
print(f'比例: {bott/plain:.4f}  (节省 {plain/bott:.1f} 倍)')
assert bott / plain < 0.1, 'bottleneck 应把参数量压到 plain 设计的 10% 以下'

# ---- 深度可分离卷积: 参数比例 = 1/Cout + 1/k^2 ----
def depthwise_separable_params(cin, cout, k):
    dw = k * k * cin
    pw = cin * cout
    return dw + pw, dw, pw

cin = cout = 256; k = 3
std_p = conv_params(cin, cout, k)
dsp_p, dw_p, pw_p = depthwise_separable_params(cin, cout, k)
print(f'\n标准卷积参数量: {std_p:,}')
print(f'深度可分离参数量: {dsp_p:,} (depthwise={dw_p:,} + pointwise={pw_p:,})')
print(f'比例: {dsp_p/std_p:.4f}   公式 1/Cout+1/k^2 = {1/cout + 1/(k*k):.4f}')
assert abs(dsp_p / std_p - (1 / cout + 1 / (k * k))) < 1e-9
print('\n✅ 验证：1x1 卷积不贡献感受野；bottleneck 省参数 ~17 倍；深度可分离卷积压缩比例精确等于 1/Cout+1/k^2。')

## 3 · 空洞卷积的 gridding artifact：集合论证明"有些像素永远采样不到"

In [ ]:
def sumset_after_layers(dilations, k=3):
    """k=3 核在膨胀率 r 下采样偏移量为 {-r,0,r}；多层堆叠的可达偏移量是各层偏移集合的 Minkowski 和。"""
    offsets_per_layer = [(-r, 0, r) for r in dilations]
    reachable = {0}
    for offs in offsets_per_layer:
        reachable = {a + b for a in reachable for b in offs}
    return reachable

const_dilation = sumset_after_layers([2, 2, 2])
hdc_dilation = sumset_after_layers([1, 2, 5])

all_positions_const = set(range(min(const_dilation), max(const_dilation) + 1))
gaps_const = sorted(all_positions_const - const_dilation)
all_positions_hdc = set(range(min(hdc_dilation), max(hdc_dilation) + 1))
gaps_hdc = sorted(all_positions_hdc - hdc_dilation)

print('恒定 dilation=(2,2,2): 可达偏移量=', sorted(const_dilation))
print('  理论感受野范围:', min(const_dilation), '~', max(const_dilation))
print('  从未被采样到的位置(gridding 空洞):', gaps_const, ' 空洞数=', len(gaps_const))
print()
print('HDC dilation=(1,2,5): 可达偏移量=', sorted(hdc_dilation))
print('  从未被采样到的位置:', gaps_hdc, ' 空洞数=', len(gaps_hdc))

assert len(gaps_const) == 6, '恒定膨胀率下，理论感受野内应有 6 个位置从未被采样到'
assert len(gaps_hdc) == 0, 'HDC (1,2,5) 方案应完全覆盖理论感受野，零空洞'
print('\n✅ 验证：连续用同一膨胀率会在理论感受野内留下规律性空洞；HDC 的锯齿膨胀率方案能补上这些空洞。')

## 4 · BN vs GroupNorm 在小 batch 下的统计稳定性

In [ ]:
C, H, W = 16, 8, 8
eps = 1e-5
x_target = rng.standard_normal((C, H, W)).astype(np.float64)   # 固定的目标样本，反复塞进不同 batch

def sample_other(r):
    """80% 概率是普通样本 N(0,1)，20% 概率是"离群样本"(比如夜间强噪声帧)，方差大 10 倍。"""
    if r.random() < 0.2:
        return r.standard_normal((C, H, W)) * np.sqrt(10.0)
    return r.standard_normal((C, H, W))

def bn_normalize_target(batch):
    """batch: (N,C,H,W)，第0个是 target。返回 target 的 BN 输出在各通道上的空间均值 (C,)。"""
    mean = batch.mean(axis=(0, 2, 3))
    var = batch.var(axis=(0, 2, 3))
    normed = (batch[0] - mean[:, None, None]) / np.sqrt(var[:, None, None] + eps)
    return normed.mean(axis=(1, 2))

T = 200
bn_vars = {}
for N in [2, 4, 8, 32, 128]:
    outs = []
    for _ in range(T):
        others = np.stack([sample_other(rng) for _ in range(N - 1)], axis=0) if N > 1 else np.zeros((0, C, H, W))
        batch = np.concatenate([x_target[None], others], axis=0)
        outs.append(bn_normalize_target(batch))
    outs = np.array(outs)
    bn_vars[N] = outs.var(axis=0).mean()
    print(f'N={N:>4}  BN 输出方差(跨 batch 组成) = {bn_vars[N]:.6f}')

Ns = [2, 4, 8, 32, 128]
assert all(bn_vars[Ns[i]] > bn_vars[Ns[i+1]] for i in range(len(Ns)-1)), 'BN 的统计不稳定性应随 batch 增大而单调下降'
assert bn_vars[2] / bn_vars[128] > 10, '小 batch 的抖动应比大 batch 明显大一个数量级以上'
print('\n✅ 验证：BN 统计量的抖动随 batch size 减小而急剧增大——这正是检测/分割任务小 batch 下 BN 出问题的数值证据。')

In [ ]:
def groupnorm_forward(x, num_groups, eps=1e-5):
    """x: (C,H,W)，单个样本，与 batch 里其他样本完全无关。"""
    C_, H_, W_ = x.shape
    G = num_groups
    s = x.reshape(G, C_ // G, H_, W_)
    mean = s.mean(axis=(1, 2, 3), keepdims=True)
    var = s.var(axis=(1, 2, 3), keepdims=True)
    return ((s - mean) / np.sqrt(var + eps)).reshape(C_, H_, W_)

def layernorm_forward(x, eps=1e-5):
    mean = x.mean()
    var = x.var()
    return (x - mean) / np.sqrt(var + eps)

def instancenorm_forward(x, eps=1e-5):
    """逐通道，只在 (H,W) 上统计。"""
    mean = x.mean(axis=(1, 2), keepdims=True)
    var = x.var(axis=(1, 2), keepdims=True)
    return (x - mean) / np.sqrt(var + eps)

# GN 的可复现性：与 batch 组成无关
gn_out1 = groupnorm_forward(x_target, num_groups=4)
gn_out2 = groupnorm_forward(x_target, num_groups=4)
assert np.allclose(gn_out1, gn_out2), 'GN 只依赖自身样本，重复计算必须完全一致'

# GN 的两个特例：G=1 等价 LN；G=C 等价 IN
gn_g1 = groupnorm_forward(x_target, num_groups=1)
ln_out = layernorm_forward(x_target)
gn_gC = groupnorm_forward(x_target, num_groups=C)
in_out = instancenorm_forward(x_target)

assert np.allclose(gn_g1, ln_out, atol=1e-9), 'G=1 时 GroupNorm 应完全等价于 LayerNorm'
assert np.allclose(gn_gC, in_out, atol=1e-9), 'G=C 时 GroupNorm 应完全等价于 InstanceNorm'
print('GN(G=1) 与 LN 是否完全一致:', np.allclose(gn_g1, ln_out, atol=1e-9))
print('GN(G=C) 与 IN 是否完全一致:', np.allclose(gn_gC, in_out, atol=1e-9))
print('\n✅ 验证：GroupNorm 是 LN(G=1) 与 IN(G=C) 的一般化，且对 batch 组成完全不敏感（方差恒为0）。')

## 5 · 从零实现 Scaled Dot-Product Attention：验证 √d_k 缩放的必要性

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, scale=True):
    d_k = Q.shape[-1]
    scores = Q @ K.T
    if scale:
        scores = scores / np.sqrt(d_k)
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

n, d_k = 20, 512
r2 = np.random.default_rng(0)
Q = r2.standard_normal((n, d_k))
K = r2.standard_normal((n, d_k))
V = r2.standard_normal((n, d_k))

out_unscaled, w_unscaled = scaled_dot_product_attention(Q, K, V, scale=False)
out_scaled, w_scaled = scaled_dot_product_attention(Q, K, V, scale=True)

raw_scores = Q @ K.T
scaled_scores = raw_scores / np.sqrt(d_k)

print('未缩放 QK^T 分数 std:', raw_scores.std(), '  理论 sqrt(d_k)=', np.sqrt(d_k))
print('缩放后分数 std      :', scaled_scores.std(), '  理论 ~1')

max_p_unscaled = w_unscaled.max(axis=-1).mean()
max_p_scaled = w_scaled.max(axis=-1).mean()
print('未缩放 softmax 最大权重均值(越接近1越"饱和"):', max_p_unscaled)
print('缩放后   softmax 最大权重均值:', max_p_scaled)

grad_scale_unscaled = (w_unscaled * (1 - w_unscaled)).mean()
grad_scale_scaled = (w_scaled * (1 - w_scaled)).mean()
print('未缩放 softmax 梯度尺度 mean(p(1-p)):', grad_scale_unscaled)
print('缩放后   softmax 梯度尺度 mean(p(1-p)):', grad_scale_scaled)

assert np.allclose(weights_sum := w_scaled.sum(axis=-1), np.ones(n)), '每行注意力权重必须归一化到 1'
assert raw_scores.std() > 15, '未缩放分数的标准差应接近 sqrt(d_k)（明显偏大）'
assert scaled_scores.std() < 2.0, '缩放后分数标准差应回落到接近 1'
assert max_p_unscaled > 0.7, '未缩放时 softmax 应明显饱和（接近 one-hot）'
assert max_p_scaled < 0.4, '缩放后 softmax 应保持较"软"的分布'
assert grad_scale_scaled > grad_scale_unscaled * 5, '缩放后 softmax 的梯度尺度应显著大于未缩放（至少 5 倍）'
print('\n✅ 验证：不除以 sqrt(d_k)，分数方差随维度线性增长导致 softmax 饱和、梯度趋于消失；'
      '缩放后梯度尺度恢复了约 10 倍。')

## 6 · 三类位置编码的外推行为对比

In [ ]:
d_pe = 8  # 偶数，RoPE 按 2 维一组旋转

def sinusoidal_pe(pos, d=8, base=10000.0):
    i = np.arange(d)
    angle = pos / (base ** (2 * (i // 2) / d))
    pe = np.zeros(d)
    pe[0::2] = np.sin(angle[0::2])
    pe[1::2] = np.cos(angle[1::2])
    return pe

def rope(x, pos, base=10000.0):
    d = x.shape[-1]
    out = x.copy()
    for j in range(0, d, 2):
        theta = pos / (base ** (j / d))
        c, s = np.cos(theta), np.sin(theta)
        x0, x1 = x[j], x[j + 1]
        out[j] = x0 * c - x1 * s
        out[j + 1] = x0 * s + x1 * c
    return out

# ① 固定正弦：任意 pos 都能直接算，即便远超训练时见过的最大长度
pe_far = sinusoidal_pe(100_000, d_pe)
assert np.all(np.isfinite(pe_far)), '正弦位置编码在超远位置仍应给出有限值'
print('sinusoidal PE 在 pos=100000 处依然可算，示例:', pe_far[:4])

# ② 可学习绝对位置嵌入：固定表，超出训练时的最大长度直接不存在
max_len = 512
learned_table = rng.standard_normal((max_len, d_pe)) * 0.02
try:
    _ = learned_table[100_000]
    can_extrapolate = True
except IndexError:
    can_extrapolate = False
print('learned absolute PE 能否直接取到 pos=100000 的向量:', can_extrapolate)
assert can_extrapolate is False, '可学习绝对位置编码架构上就不支持超出表长的位置'

# ③ 类 RoPE：只依赖相对位置差，即便绝对位置远超"训练范围"也精确成立
r3 = np.random.default_rng(1)
q_vec, k_vec = r3.standard_normal(d_pe), r3.standard_normal(d_pe)
dot_near = rope(q_vec, 5) @ rope(k_vec, 8)          # 相对位置差 = 3
dot_far = rope(q_vec, 1005) @ rope(k_vec, 1008)     # 相对位置差同样 = 3，但绝对位置远超常见训练长度
dot_diff_offset = rope(q_vec, 5) @ rope(k_vec, 9)   # 相对位置差 = 4，应该给出不同的点积

print(f'RoPE: 相对位置差=3 时 dot(近)={dot_near:.6f}  dot(远)={dot_far:.6f}  是否几乎相等: {np.isclose(dot_near, dot_far, atol=1e-8)}')
print(f'RoPE: 相对位置差=4 时 dot={dot_diff_offset:.6f}  (应与上面不同)')

assert np.isclose(dot_near, dot_far, atol=1e-8), 'RoPE 的点积应严格只依赖相对位置差，与绝对位置无关'
assert not np.isclose(dot_near, dot_diff_offset, atol=1e-4), '相对位置差不同时，点积应不同'
print('\n✅ 验证：三类位置编码里，只有固定正弦与类 RoPE 能"算"到训练时未见过的位置；'
      '可学习绝对位置编码架构上有硬性长度上限；RoPE 的相对不变性是精确的数学恒等式，不是近似。')

## ✏️ 练习 1：卷积参数量与 FLOPs 计算器

实现 `conv_flops_and_params(cin, cout, k, hout, wout, bias=False)`，
返回 `(params, macs, flops)`，其中 `flops = 2 * macs`。

In [ ]:
def conv_flops_and_params(cin, cout, k, hout, wout, bias=False):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
params, macs, flops = conv_flops_and_params(3, 64, 7, 112, 112, bias=False)
assert params == 9408, params
assert macs == 118_013_952, macs
assert flops == 2 * macs
params2, macs2, flops2 = conv_flops_and_params(64, 64, 3, 56, 56, bias=True)
assert params2 == 3 * 3 * 64 * 64 + 64
assert macs2 == 3 * 3 * 64 * 64 * 56 * 56
print(f'ResNet conv1: params={params:,}  MACs={macs:,}  FLOPs={flops:,}')
print(f'3x3(64->64,56x56,带bias): params={params2:,}  MACs={macs2:,}  FLOPs={flops2:,}')
print('✅ 练习 1 通过。')

## ✏️ 练习 2：带膨胀率的感受野递推

实现 `receptive_field_dilated(layers)`，`layers` 元素为 `(k, s, dilation)`。
膨胀率为 $d$ 时，核的"有效跨度"变成 $k_{\text{eff}} = k + (k-1)(d-1)$，
其余递推与普通感受野公式完全一样（用 $k_{\text{eff}}$ 替换 $k$）。

In [ ]:
def receptive_field_dilated(layers):
    # TODO: 返回 [(RF, jump), ...]，RF_0=1, jump_0=1
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# 膨胀率恒为1时应退化为普通感受野公式
seq_d1 = receptive_field_dilated([(3, 1, 1), (3, 1, 1), (3, 1, 1)])
assert seq_d1[-1][0] == 7, seq_d1

# 单层 3x3, dilation=2: k_eff = 3+(3-1)*(2-1) = 5, RF = 1+(5-1)*1 = 5
seq_d2 = receptive_field_dilated([(3, 1, 2)])
assert seq_d2[-1][0] == 5, seq_d2

# 三层 3x3, dilation 分别为 1,2,5 (HDC): 逐层核跨度为 3,5,11
seq_hdc = receptive_field_dilated([(3, 1, 1), (3, 1, 2), (3, 1, 5)])
print('HDC (1,2,5) 三层的 RF 序列:', seq_hdc)
assert seq_hdc[-1][0] == 1 + 2 + 4 + 10   # 1 + (3-1) + (5-1) + (11-1) = 17
print('✅ 练习 2 通过：膨胀率通过"有效跨度"进入同一套感受野递推公式。')

## ✏️ 练习 3：GroupNorm 与 LN / IN 的等价关系

实现 `is_gn_equivalent_to(x, kind)`，`kind` 为 `'ln'` 或 `'in'`：
用 `groupnorm_forward` 分别以 `num_groups=1`（应等价 LN）和 `num_groups=C`（应等价 IN）计算，
返回布尔值表示是否与对应的参考实现（`layernorm_forward` / `instancenorm_forward`）数值一致。

In [ ]:
def is_gn_equivalent_to(x, kind):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert is_gn_equivalent_to(x_target, 'ln') is True
assert is_gn_equivalent_to(x_target, 'in') is True
x_other = rng.standard_normal((C, H, W))
assert is_gn_equivalent_to(x_other, 'ln') is True   # 对任意样本都应成立，不是只对 x_target 恰好成立
print('✅ 练习 3 通过：GroupNorm(G=1)==LayerNorm，GroupNorm(G=C)==InstanceNorm，对任意输入恒成立。')

## ✏️ 练习 4：从零实现并验证 Scaled Dot-Product Attention 的基本性质

实现 `attention_row_stochastic_check(Q, K, V, scale=True)`，
复用第 5 节的 `scaled_dot_product_attention`，返回 `(weights 每行和是否都为1, 输出形状是否等于V的形状)`
两个布尔值。

In [ ]:
def attention_row_stochastic_check(Q, K, V, scale=True):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
r4 = np.random.default_rng(2)
Qs, Ks, Vs = r4.standard_normal((7, 16)), r4.standard_normal((7, 16)), r4.standard_normal((7, 16))
rows_ok, shape_ok = attention_row_stochastic_check(Qs, Ks, Vs, scale=True)
assert rows_ok is True
assert shape_ok is True
rows_ok2, shape_ok2 = attention_row_stochastic_check(Qs, Ks, Vs, scale=False)
assert rows_ok2 is True, '不管缩不缩放，softmax 输出都必须每行归一化到 1'
print('✅ 练习 4 通过：无论是否缩放，注意力权重矩阵都必须行归一化，输出形状必须与 V 一致。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def conv_flops_and_params(cin, cout, k, hout, wout, bias=False):
    params = k * k * cin * cout + (cout if bias else 0)
    macs = k * k * cin * cout * hout * wout
    flops = 2 * macs
    return params, macs, flops

In [ ]:
# 练习 2 参考答案
def receptive_field_dilated(layers):
    rf, jump = 1, 1
    out = [(rf, jump)]
    for k, s, d in layers:
        k_eff = k + (k - 1) * (d - 1)
        rf = rf + (k_eff - 1) * jump
        jump = jump * s
        out.append((rf, jump))
    return out

In [ ]:
# 练习 3 参考答案
def is_gn_equivalent_to(x, kind):
    if kind == 'ln':
        C_ = x.shape[0]
        return bool(np.allclose(groupnorm_forward(x, num_groups=1), layernorm_forward(x), atol=1e-9))
    elif kind == 'in':
        C_ = x.shape[0]
        return bool(np.allclose(groupnorm_forward(x, num_groups=C_), instancenorm_forward(x), atol=1e-9))
    raise ValueError(kind)

In [ ]:
# 练习 4 参考答案
def attention_row_stochastic_check(Q, K, V, scale=True):
    out, weights = scaled_dot_product_attention(Q, K, V, scale=scale)
    rows_ok = bool(np.allclose(weights.sum(axis=-1), np.ones(weights.shape[0])))
    shape_ok = bool(out.shape == V.shape)
    return rows_ok, shape_ok

---
## 🧪 真实工程胶囊：架构选型与常见踩雷速查

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# A. 卷积口算速查（面试可直接背的版本）
# ══════════════════════════════════════════════════════════════════════
# 参数量  = k^2 * Cin * Cout                       (与分辨率无关)
# MACs    = 参数量 * Hout * Wout                    FLOPs = 2 * MACs
# RF_l    = RF_(l-1) + (k_l-1)*jump_(l-1)           jump_l = jump_(l-1)*s_l
# 1x1 卷积: k=1 时对 RF 无贡献，只做通道变换/混合/廉价加非线性
# 深度可分离: 参数比例 = 1/Cout + 1/k^2 （通常远小于1，但depthwise是访存受限，
#            实际部署加速比 < 理论 FLOPs 加速比）

# ══════════════════════════════════════════════════════════════════════
# B. 归一化选型速查
# ══════════════════════════════════════════════════════════════════════
# □ 分类/大 batch 训练           -> BatchNorm（训练用 batch 统计，推理用 running stats）
# □ 检测/分割/小 batch(1-4)       -> GroupNorm（或多卡 SyncBN）
# □ Transformer（任意模态）       -> LayerNorm（不依赖 batch 维度，变长序列友好）
# □ 风格迁移/需要保留单样本对比度   -> InstanceNorm
# 记忆锚点: GroupNorm(G=1)==LayerNorm, GroupNorm(G=C)==InstanceNorm

# ══════════════════════════════════════════════════════════════════════
# C. Attention 常见踩雷点
# ══════════════════════════════════════════════════════════════════════
# 坑1: 忘记除以 sqrt(d_k) -> 训练初期 loss 不降或直接 NaN（softmax 饱和梯度消失）
# 坑2: padding token 忘记 mask -> attention 会"看到"填充位置，注意力权重被污染
# 坑3: 可学习绝对位置编码 + 推理时序列变长 -> 直接 index out of range
# 坑4: FlashAttention 只优化显存/延迟常数，不改变 O(n^2) 的理论 FLOPs，
#      长序列的根本瓶颈仍需架构层面（稀疏/线性/窗口注意力）来解决

# ══════════════════════════════════════════════════════════════════════
# D. CNN vs Transformer 选型的一句话决策树
# ══════════════════════════════════════════════════════════════════════
# 数据量小 / 需要快速收敛 / 部署工具链要求成熟  -> CNN（或 CNN backbone + 轻量 attention）
# 数据量大 / 需要长距离建模 / 能接受更长训练与更新的部署工具链 -> Transformer / 混合架构
# TSR 等车载检测任务的现实选择通常是: CNN backbone 提特征 + 少量 attention 做跨尺度融合
# (呼应 C53 的 RTMDet/RT-DETR 混合设计)

# ══════════════════════════════════════════════════════════════════════
# E. 与本课程其他部分的分工（别重复准备）
# ══════════════════════════════════════════════════════════════════════
# · 卷积反向传播的完整数学推导                 -> C18 模块 02-03（本课不重复）
# · Transformer 完整结构与训练细节             -> C00 模块 01/03
# · 大核卷积的有效感受野实测、结构重参数化       -> C53 模块 01/03
# · 可变形注意力、object query、匈牙利匹配      -> C54 模块 01/03/04
# · 优化器、初始化、损失函数                   -> C64 模块 02（上一站）
'''
print(RECIPE)
for token in ['sqrt(d_k)', 'GroupNorm(G=1)', 'FlashAttention', 'C18 模块 02-03', 'C54 模块 01/03/04']:
    assert token in RECIPE, token
print('✅ 检查单覆盖：卷积口算 / 归一化选型 / attention 踩雷 / CNN-Transformer 决策树 / 课程分工')

### 小结

- **参数量只看卷积核和通道数，FLOPs 还要再乘输出分辨率**——这一句话就能接住"参数少但算力大"的追问。
  **1×1 卷积不贡献感受野**，它的价值在通道变换/混合/廉价加非线性；bottleneck 结构能把参数压到不到 1/10。
- **空洞卷积的膨胀率不能每层都一样**——连续用同一个膨胀率会在理论感受野内留下永远采不到的"棋盘空洞"，
  必须用锯齿状的膨胀率组合（如 1,2,5）填补。
- **归一化家族只有一个区别：在哪些维度上求统计量**。BN 依赖 batch 维度，小 batch/检测任务下统计量抖动剧烈；
  GroupNorm 是 LN(G=1) 与 IN(G=C) 的一般化，对 batch 组成完全不敏感——这是它替代 BN 的根本原因。
- **除以 $\sqrt{d_k}$ 不是经验技巧，是精确的方差控制**：不缩放会让 softmax 饱和、梯度消失约 10 倍。
- **三类位置编码的外推行为完全不同**：可学习绝对编码架构上有硬性长度上限，固定正弦和类 RoPE 都能算到训练时未见过的位置，
  RoPE 的相对位置不变性是精确的数学恒等式。
- **CNN vs Transformer 的核心取舍是归纳偏置 vs 数据量**：先验知识在数据稀缺时值钱，在数据充裕时可能变成枷锁。

下一站：**模块 04 · 评估、概率与统计问答** —— 指标全家桶、校准、置信区间、A/B 测试与常见统计陷阱。